# Notebook 06 — Feature Importance & Interpretability

**Project:** Telco Customer Churn Prediction — IIT Roorkee Capstone

This notebook:
1. Plots **XGBoost built-in importance** (over one-hot encoded feature names)
2. Computes **permutation importance** over the raw original column names
3. Explains why the two views differ and when each is more useful
4. Maps top features to concrete business retention actions

**Prerequisite:** Run Notebook 05 first (creates the tuned `best_model.joblib`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

%matplotlib inline
sns.set_theme(style='whitegrid', context='talk')

DATA_PATH   = Path('..') / 'data' / 'raw' / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
MODEL_PATH  = Path('..') / 'outputs' / 'models' / 'best_model.joblib'
FIGURES_DIR = Path('..') / 'outputs' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
print('Ready.')

In [ ]:
df = pd.read_csv(DATA_PATH)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges']).reset_index(drop=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df = df.drop(columns=['customerID'])

y = df['Churn']
X = df.drop(columns=['Churn'])

_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

pipeline = joblib.load(MODEL_PATH)
print(f'Loaded model: {type(pipeline.named_steps["model"]).__name__}')
print(f'Test set: {X_test.shape}')

## Why Two Views of Importance?

| Method | Input features | Speed | Limitation |
|--------|---------------|-------|------------|
| **XGB built-in** (feature_importances_) | One-hot encoded | Instant | Reflects how trees were *built*; biased toward high-cardinality features |
| **Permutation importance** | Raw original columns | ~30s | More reliable for business interpretation; shuffles real columns |

For the **presentation**, we use permutation importance because it speaks in the language of the original dataset — "Contract" matters, not "cat__Contract_Two year".

## 1. XGBoost Built-in Feature Importance (Encoded Features)

XGBoost records the **average gain** each feature contributes when it is used in a split. This is measured over the one-hot encoded columns, so each dummy column is separate.

In [ ]:
model = pipeline.named_steps['model']
preprocessor = pipeline.named_steps['preprocess']

if not hasattr(model, 'feature_importances_'):
    print(f'Model {type(model).__name__} does not have feature_importances_. Skipping.')
else:
    feat_names = preprocessor.get_feature_names_out().tolist()
    importance_df = pd.DataFrame({
        'feature':    feat_names,
        'importance': model.feature_importances_,
    }).sort_values('importance', ascending=False)

    top_n = 20
    top_df = importance_df.head(top_n)
    print(f'Total encoded features: {len(feat_names)}')
    print(f'\nTop {top_n} features by XGB gain importance:')
    print(top_df.to_string(index=False))

In [ ]:
if hasattr(model, 'feature_importances_'):
    top15 = importance_df.head(15)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top15['feature'][::-1], top15['importance'][::-1], color='#4C72B0')
    ax.set_xlabel('Gain importance')
    ax.set_title('Top 15 encoded features — XGBoost built-in importance')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '07_xgb_feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/figures/07_xgb_feature_importance.png')

## 2. Permutation Importance (Raw Original Columns)

Algorithm:
1. Score the full pipeline on the test set (baseline ROC-AUC)
2. For each column: shuffle its values randomly, score again
3. Importance = baseline − shuffled score (drop in performance)
4. Repeat 10 times and report mean ± std

Because we pass the **full Pipeline**, shuffling happens on the raw column, and preprocessing is re-applied — giving us importance in terms of the original dataset.

In [ ]:
print('Running permutation importance (10 repeats × 19 features = ~2 min)...')
perm_result = permutation_importance(
    pipeline, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    scoring='roc_auc',
    n_jobs=1,
)

perm_df = pd.DataFrame({
    'feature': X_test.columns,
    'mean':    perm_result.importances_mean,
    'std':     perm_result.importances_std,
}).sort_values('mean', ascending=False)

print('\nTop 10 features by permutation importance:')
print(perm_df.head(10).round(5).to_string(index=False))

In [ ]:
top15_perm = perm_df.head(15)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(
    top15_perm['feature'][::-1],
    top15_perm['mean'][::-1],
    xerr=top15_perm['std'][::-1],
    capsize=4, color='#DD8452',
)
ax.set_xlabel('Drop in ROC-AUC when column is shuffled')
ax.set_title('Top 15 raw features — Permutation importance (10 repeats)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '08_permutation_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/figures/08_permutation_importance.png')

## 3. Compare Both Views

In [ ]:
print('=== Top features by permutation importance (raw columns) ===')
for _, row in perm_df.head(8).iterrows():
    print(f'  {row["feature"]:>20}  drop={row["mean"]:+.5f}  std={row["std"]:.5f}')

if hasattr(model, 'feature_importances_'):
    print('\n=== Top 8 by XGB built-in (encoded) ===')
    for _, row in importance_df.head(8).iterrows():
        print(f'  {row["feature"]:>35}  gain={row["importance"]:.5f}')

print('\nNote: XGB built-in splits "Contract" into two dummies;')
print('permutation importance treats it as one column — more intuitive for business.')

## 4. Business Interpretation — Feature → Retention Action

Each top feature maps to a specific campaign a telecom retention team can run.

In [ ]:
action_map = {
    'tenure':          'High-risk window: first 6 months. Invest in onboarding (dedicated support call at 30/60/90 days).',
    'Contract':        'Month-to-month churn rate = 43%. Push 12/24-month contracts with first-month discount.',
    'InternetService': 'Fiber-optic churn = 42% despite higher bills. Investigate NPS and service quality for this segment.',
    'MonthlyCharges':  'Churners pay ~$10/month more. Flag high-bill customers for proactive price-lock or bundle offers.',
    'OnlineSecurity':  'Customers without OnlineSecurity churn more. Offer as a free 6-month retention add-on.',
    'TechSupport':     'Same as OnlineSecurity — bundle free TechSupport for at-risk customers.',
    'PaymentMethod':   'Electronic check → 45% churn. Migrate to auto-pay with a one-month bill credit.',
    'TotalCharges':    'Correlated with tenure — long-tenured customers have paid more and are more loyal.',
}

top_features = perm_df.head(8)['feature'].tolist()
print('=== Feature → Business Retention Action ===')
for i, feat in enumerate(top_features, 1):
    action = action_map.get(feat, 'Investigate further with domain expert.')
    print(f'\n{i}. {feat}')
    print(f'   {action}')

In [ ]:
# Churn rate for the top 3 categorical features (Contract, InternetService, PaymentMethod)
df_raw = pd.read_csv(DATA_PATH)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col in zip(axes, ['Contract', 'InternetService', 'PaymentMethod']):
    rates = df_raw.groupby(col)['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).sort_values(ascending=False)
    bars = ax.bar(rates.index, rates.values, color='#DD8452')
    ax.set_title(f'Churn rate by {col}')
    ax.set_ylabel('Churn rate (%)')
    ax.tick_params(axis='x', rotation=25)
    for bar, v in zip(bars, rates.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.5,
                f'{v:.1f}%', ha='center', fontsize=10)

fig.suptitle('Top categorical drivers — churn rate by category', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP (SHapley Additive exPlanations) — optional deeper analysis
# Uncomment if shap is installed: uv add shap

# import shap
# explainer = shap.TreeExplainer(pipeline.named_steps['model'])
# X_test_encoded = pipeline.named_steps['preprocess'].transform(X_test)
# shap_values = explainer.shap_values(X_test_encoded)
# shap.summary_plot(shap_values, X_test_encoded,
#                   feature_names=pipeline.named_steps['preprocess'].get_feature_names_out())

print('SHAP analysis is available but commented out to avoid extra dependency.')
print('To enable: uv add shap  then uncomment the cells above.')

## Summary

- **XGBoost built-in importance** shows which one-hot dummy columns were used in splits (fast but harder to interpret)
- **Permutation importance** shows which original columns matter most to the model's discriminative ability on the test set (preferred for business communication)
- Top drivers: **tenure, Contract, InternetService, MonthlyCharges, OnlineSecurity**
- Each maps to a concrete retention campaign — these are actionable insights, not just model artefacts

**Next:** [07_threshold_tuning.ipynb](07_threshold_tuning.ipynb)